In [ ]:
# Google Colab Setup
# Run this cell first to install dependencies
!pip install -q \
    langchain langchain-core langchain-openai langchain-anthropic \
    langchain-google-genai langchain-google-vertexai langchain-community \
    langchain-tavily langchain-text-splitters langchain-model-profiles \
    langgraph langsmith python-dotenv \
    mcp langchain-mcp-adapters \
    pypdf ipywidgets tqdm tavily


In [ ]:
# API Key Setup
# In Google Colab: open the Secrets panel (🔑 icon in the left sidebar),
# add each key by name, and enable notebook access.
# Locally: keys are loaded from your .env file automatically.
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    for key in ["ANTHROPIC_API_KEY", "GOOGLE_API_KEY", "TAVILY_API_KEY",
                "LANGSMITH_API_KEY", "LANGSMITH_TRACING", "LANGSMITH_PROJECT"]:
        try:
            val = userdata.get(key)
            if val:
                os.environ[key] = val
        except Exception:
            pass
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()


In [ ]:
from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "blue"
    least_favourite_colour: str = "yellow"

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    context_schema=ColourContext  
)

In [ ]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

In [ ]:
from pprint import pprint

pprint(response)

## Accessing Context

In [ ]:
from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the favourite colour of the user"""
    return runtime.context.favourite_colour

@tool
def get_least_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

In [ ]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext
)

In [ ]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

pprint(response)

In [ ]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(favourite_colour="green")
)

pprint(response)